# LeetCode 237 — Delete Node in a Linked List bằng Python

Link đề: https://leetcode.com/problems/delete-node-in-a-linked-list/ · Độ khó: **Medium** · Chủ đề: Linked List

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, x):
#         self.val = x
#         self.next = None
class Solution:
    def deleteNode(self, node):
        """
        :type node: ListNode
        :rtype: void Do not return anything, modify node in-place instead.
        """
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code: ta dựng danh sách liên kết, lấy nút theo vị trí, gọi hàm rồi so sánh toàn bộ danh sách với kết quả mong đợi, đồng thời đối chiếu với cách xóa "bình thường" khi có đầu danh sách.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ta chỉ được đưa **đúng nút cần xóa** (không có `head`, và nút đó **không phải nút cuối**); phải "xóa" nó sao cho giá trị đó biến mất, độ dài giảm 1 và thứ tự các giá trị còn lại giữ nguyên.
2. Biết **tự dựng danh sách liên kết** (`ListNode`), chuyển qua lại giữa **list Python và danh sách liên kết**, và lấy nút theo vị trí để test.
3. Hiểu vì sao cách xóa thông thường (`prev.next = node.next`) **không dùng được** (không có `prev`), và nhận ra **mẹo**: không xóa chính nút đó mà **sao chép giá trị của nút kế tiếp vào nó rồi xóa nút kế tiếp**.
4. Đi từ **xóa thông thường khi có `head` (baseline)** → **mẹo sao chép nút kế tiếp (`O(1)`)** → **dịch giá trị về trái đến cuối rồi cắt đuôi (`O(n)`)** → **sao chép cả `val` và `next` bằng phép gán song song** → **dùng nút giả `dummy` khi có đầu danh sách**.
5. Tránh các lỗi kinh điển: gán lại biến cục bộ (`node = node.next` không làm gì), chỉ nối `node.next = node.next.next` mà quên sao chép giá trị (xóa nhầm nút kế tiếp), sai thứ tự gán, và hiểu tại sao nút cuối không được phép.

## 2. Đề bài & ví dụ

Có một danh sách liên kết đơn `head` và ta muốn xóa một nút `node` trong đó. Bạn được cung cấp **chỉ nút cần xóa** `node`. Bạn **không** được cấp `head`.

Mọi giá trị trong danh sách đều **khác nhau**, và `node` được đảm bảo **không phải nút cuối**.

"Xóa" ở đây **không** có nghĩa là giải phóng bộ nhớ. Ý là:

- giá trị của `node` không còn tồn tại trong danh sách;
- số nút của danh sách giảm đi 1;
- các giá trị đứng **trước** `node` giữ nguyên thứ tự;
- các giá trị đứng **sau** `node` giữ nguyên thứ tự.

Cách kiểm tra tùy chỉnh: người chấm dựng danh sách `head`, chọn nút `node`, gọi hàm của bạn và so sánh **toàn bộ danh sách** sau khi gọi.

**Ví dụ 1**

- Input: `head = [4,5,1,9]`, `node = 5`
- Output: `[4,1,9]`
- Giải thích: bạn được nút thứ hai (giá trị `5`); sau khi gọi, danh sách là `4 -> 1 -> 9`.

**Ví dụ 2**

- Input: `head = [4,5,1,9]`, `node = 1`
- Output: `[4,5,9]`
- Giải thích: bạn được nút thứ ba (giá trị `1`); danh sách thành `4 -> 5 -> 9`.

**Ràng buộc:** số nút trong khoảng `[2, 1000]`; `-1000 <= Node.val <= 1000`; mọi giá trị đều khác nhau; nút cần xóa nằm trong danh sách và **không phải nút đuôi**.

In [1]:
from typing import List, Optional
import random


class ListNode:
    def __init__(self, x):
        self.val = x
        self.next = None


def from_list(values):
    """list Python -> danh sách liên kết (trả về head, hoặc None nếu rỗng)."""
    head = tail = None
    for v in values:
        node = ListNode(v)
        if head is None:
            head = tail = node
        else:
            tail.next = node
            tail = node
    return head


def to_list(head, limit=5000):
    """Danh sách liên kết -> list Python (có giới hạn để phát hiện chu trình do bug)."""
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
        assert len(out) <= limit, "danh sách quá dài: có thể có chu trình"
    return out


def node_at(head, index):
    """Lấy nút ở vị trí index (đếm từ 0)."""
    for _ in range(index):
        head = head.next
    return head


examples = [
    ([4, 5, 1, 9], 1, [4, 1, 9]),     # node = nút thứ hai (giá trị 5)
    ([4, 5, 1, 9], 2, [4, 5, 9]),     # node = nút thứ ba (giá trị 1)
]
for values, idx, expected in examples:
    head = from_list(values)
    print(f"head={to_list(head)}, node = nút ở vị trí {idx} (giá trị {node_at(head, idx).val}), kỳ vọng {expected}")

head=[4, 5, 1, 9], node = nút ở vị trí 1 (giá trị 5), kỳ vọng [4, 1, 9]
head=[4, 5, 1, 9], node = nút ở vị trí 2 (giá trị 1), kỳ vọng [4, 5, 9]


## 3. Kiến thức nền

### 3.1. Nút danh sách liên kết và cách duyệt

Mỗi nút `ListNode` có `val` (giá trị) và `next` (nút kế tiếp, hoặc `None` ở nút cuối). Danh sách được xác định bởi `head`. Để duyệt: `while cur: ...; cur = cur.next`. Hàm `from_list` và `to_list` ở trên giúp ta đổi qua lại với list Python để dễ in và so sánh.

In [2]:
head = from_list([4, 5, 1, 9])
cur = head
while cur:
    print(cur.val, "->", end=" ")
    cur = cur.next
print("None")
print(to_list(head))                      # [4, 5, 1, 9]
print(to_list(from_list([])))             # []  (danh sách rỗng)

4 -> 5 -> 1 -> 9 -> None
[4, 5, 1, 9]
[]


### 3.2. Xóa nút thông thường: cần nút đứng TRƯỚC

Thông thường để xóa `node` ta cần nút trước nó `prev` rồi đặt `prev.next = node.next`. Danh sách đơn **chỉ trỏ tiến**, nên từ `node` không thể quay lại `prev`. Đề cố ý không cho `head`, tức không thể tìm `prev` bằng cách duyệt từ đầu.

In [3]:
def delete_with_prev(head, node):
    """Xóa thông thường, CẦN head (không dùng được trong bài này)."""
    if head is node:
        return head.next
    prev = head
    while prev.next is not node:
        prev = prev.next
    prev.next = node.next
    return head


head = from_list([4, 5, 1, 9])
head = delete_with_prev(head, node_at(head, 1))
print(to_list(head))                      # [4, 1, 9]

[4, 1, 9]


### 3.3. Nút là đối tượng: có thể ghi đè nội dung của nó

Mỗi `ListNode` là một **đối tượng**; biến `node` là một **tham chiếu** tới nó. Ta có thể **ghi đè các trường** của đối tượng (`node.val = ...`, `node.next = ...`), và mọi nơi đang trỏ tới đối tượng này (ví dụ `prev.next`) sẽ thấy thay đổi. Nhưng **gán lại biến** `node = ...` chỉ đổi biến cục bộ, không đổi đối tượng nào.

In [4]:
a = ListNode(1)
b = a                        # b và a cùng tham chiếu đến một đối tượng
b.val = 99                   # ghi đè trường của đối tượng: a cũng thấy
print(a.val)                 # 99

def rebind(node):
    node = ListNode(-1)      # chỉ đổi biến cục bộ 'node', không đụng gì tới đối tượng bên ngoài

x = ListNode(5)
rebind(x)
print(x.val)                 # 5 (không đổi)

99
5


### 3.4. Gán song song và thứ tự thực hiện

`a, b = expr1, expr2` **tính cả hai biểu thức vế phải trước**, rồi gán từ trái sang phải. Còn viết hai lệnh gán tách rời thì lệnh đầu đã thay đổi trạng thái trước khi lệnh sau chạy. Chênh lệch này rất quan trọng khi các trường phụ thuộc lẫn nhau (như `node.next`).

In [5]:
a, b = 1, 2
a, b = b, a + b            # vế phải tính với a, b cũ: (2, 3)
print(a, b)                # 2 3

x, y = 1, 2
x = y                       # tách rời: x đã đổi
y = x + y                   # dùng x mới
print(x, y)                 # 2 4  (khác với gán song song)

2 3
2 4


### 3.5. Giả cho phép `dummy` (nút giả đứng trước `head`)

Khi có `head`, xóa nút đầu danh sách là trường hợp đặc biệt vì không có nút trước. Thêm một nút giả `dummy` với `dummy.next = head` biến mọi nút thành "có nút trước", nên không cần xử lý riêng. Kết quả là `dummy.next`.

In [6]:
head = from_list([4, 5, 1, 9])
dummy = ListNode(0)
dummy.next = head
prev = dummy
while prev.next and prev.next.val != 4:     # xóa nút giá trị 4 (nút đầu)
    prev = prev.next
prev.next = prev.next.next
print(to_list(dummy.next))                   # [5, 1, 9]

[5, 1, 9]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Danh sách `4 → 5 → 1 → 9`, ta được nút `5` (nút thứ hai) và phải làm nó "biến mất".

Ta không biết nút `4` đứng trước để bảo nó "bỏ qua `5`". Nhưng ta **có thể sửa nội dung của chính nút `5`**:

1. nút kế tiếp của `5` là `1`: **chép giá trị `1` vào nút `5`**: danh sách trông như `4 → 1 → 1 → 9` (nút thứ hai giờ chứa `1`);
2. bây giờ nút thứ ba (giá trị `1`) là bản sao thừa: **cho nút thứ hai nhảy qua nút thứ ba**: `node.next = node.next.next`: danh sách `4 → 1 → 9`.

Nhìn từ bên ngoài, giá trị `5` đã biến mất, độ dài giảm 1 và thứ tự các giá trị còn lại không đổi: đúng yêu cầu đề.

Vì sao cần "nút không phải đuôi"? Vì ta lấy giá trị từ `node.next`. Nếu `node` là đuôi thì `node.next` là `None` và không có gì để sao chép.

In [7]:
head = from_list([4, 5, 1, 9])
node = node_at(head, 1)
print("ban đầu          :", to_list(head))
node.val = node.next.val                       # bước 1: chép giá trị của nút kế tiếp
print("sau khi chép val  :", to_list(head))
node.next = node.next.next                     # bước 2: bỏ qua nút kế tiếp
print("sau khi nối bỏ qua:", to_list(head))

ban đầu          : [4, 5, 1, 9]
sau khi chép val  : [4, 1, 1, 9]
sau khi nối bỏ qua: [4, 1, 9]


## 5. Cách 1 — Brute Force / baseline: xóa thông thường khi **có** `head`

**Ý tưởng.** Đây không phải là lời giải hợp lệ cho đề (đề không cho `head`), nhưng là **chuẩn đối chiếu**: nếu ta có `head`, tìm nút `prev` đứng trước `node` bằng cách duyệt từ đầu rồi đặt `prev.next = node.next`. Ta dùng nó để kiểm tra kết quả của các cách không cần `head`.

In [8]:
def delete_with_head(head, node):
    """Xóa thông thường: duyệt từ head tìm prev. Trả về head mới. (Chỉ dùng làm chuẩn đối chiếu.)"""
    if head is node:                       # xóa nút đầu: head mới là node.next
        return node.next
    prev = head
    while prev.next is not node:           # O(n): tìm nút đứng trước node
        prev = prev.next
    prev.next = node.next                  # nối prev tới nút sau node
    return head


for values, idx, expected in examples:
    head = from_list(values)
    head = delete_with_head(head, node_at(head, idx))
    print(f"head={values}, xóa vị trí {idx} -> {to_list(head)} (kỳ vọng {expected})")

head=[4, 5, 1, 9], xóa vị trí 1 -> [4, 1, 9] (kỳ vọng [4, 1, 9])
head=[4, 5, 1, 9], xóa vị trí 2 -> [4, 5, 9] (kỳ vọng [4, 5, 9])


**Độ phức tạp (Cách 1).** Gọi `n` là số nút. Duyệt tìm `prev` tốn `O(n)`: **Time `O(n)`**, **Space `O(1)`**. Nhưng cần `head`, điều mà đề **không cho**.

**Nút thắt.** Đề chỉ đưa `node`. Từ `node` ta chỉ đi **tiến** được. Vậy ta phải xóa mà không tìm `prev`: thay vào đó **thay đổi nội dung của chính `node`**.

## 6. Ý tưởng thuật toán / bước đột phá

Nhìn từ ngoài, danh sách chỉ là **dãy các giá trị** theo thứ tự. Để xóa giá trị `x` ở vị trí `i` ta không nhất thiết phải bỏ **chính đối tượng nút** đó ra khỏi danh sách; ta chỉ cần danh sách sau thay đổi có dãy giá trị bằng dãy ban đầu trừ `x`.

**Mẹo:** cho nút `node` **mượn giá trị** của nút kế tiếp `nxt = node.next`, rồi **cho `node` nối thẳng tới `nxt.next`**:

```
node.val  = node.next.val
node.next = node.next.next
```

Sau hai lệnh: vị trí `i` giờ chứa giá trị của vị trí `i + 1`; nút kế tiếp cũ bị bỏ qua; các giá trị còn lại giữ nguyên thứ tự. Dãy giá trị: bỏ `x` đi, đúng yêu cầu.

Đây là cách giải `O(1)` thời gian, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

### Cách 2 — Sao chép giá trị nút kế tiếp rồi bỏ qua nó (cách của lời giải tham khảo)

**Giải thích.** Chính là hai lệnh ở mục 6. Hợp lệ vì `node.next` không rỗng (đề đảm bảo `node` không phải đuôi).

In [9]:
def delete_copy_next(node):
    node.val = node.next.val          # 1. chép giá trị của nút kế tiếp vào node
    node.next = node.next.next        # 2. bỏ qua nút kế tiếp (nay là bản sao thừa)


for values, idx, expected in examples:
    head = from_list(values)
    delete_copy_next(node_at(head, idx))
    print(f"head={values}, xóa vị trí {idx} -> {to_list(head)} (kỳ vọng {expected})")

head=[4, 5, 1, 9], xóa vị trí 1 -> [4, 1, 9] (kỳ vọng [4, 1, 9])
head=[4, 5, 1, 9], xóa vị trí 2 -> [4, 5, 9] (kỳ vọng [4, 5, 9])


In [10]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def deleteNode(self, node):
        """
        :type node: ListNode
        :rtype: void Do not return anything, modify node in-place instead.
        """
        node.val = node.next.val
        node.next = node.next.next

**Độ phức tạp Cách 2:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 3 — Dịch mọi giá trị về trái đến cuối rồi cắt đuôi

**Giải thích.** Cách "thô" hơn nhưng cùng tinh thần: từ `node`, ghi đè giá trị của từng nút bằng giá trị của nút kế tiếp, đi tới nút áp cuối; cuối cùng cắt nút đuôi (`prev.next = None`). Kết quả: các giá trị từ vị trí `i + 1` dịch trái một ô; đuôi bị bỏ. Hữu ích để thấy vì sao chỉ cần "mượn" nút kế tiếp là đủ.

In [11]:
def delete_shift_left(node):
    cur = node
    while cur.next.next is not None:        # dừng ở nút áp cuối
        cur.val = cur.next.val              # dịch giá trị về trái một ô
        cur = cur.next
    cur.val = cur.next.val                  # nút áp cuối nhận giá trị của đuôi
    cur.next = None                         # cắt đuôi


for values, idx, expected in examples:
    head = from_list(values)
    delete_shift_left(node_at(head, idx))
    print(f"head={values}, xóa vị trí {idx} -> {to_list(head)} (kỳ vọng {expected})")

head=[4, 5, 1, 9], xóa vị trí 1 -> [4, 1, 9] (kỳ vọng [4, 1, 9])
head=[4, 5, 1, 9], xóa vị trí 2 -> [4, 5, 9] (kỳ vọng [4, 5, 9])


**Độ phức tạp Cách 3:** **Time `O(n)`** (`n` ở đây là số nút từ `node` đến cuối), **Space `O(1)`**. Chậm hơn Cách 2 mà không được gì thêm, nên chỉ để minh họa.

### Cách 4 — Chép cả `val` lẫn `next` bằng gán song song

**Giải thích.** Gộp hai lệnh của Cách 2 thành một phép gán song song: `node.val, node.next = node.next.val, node.next.next`. Vì vế phải được tính xong **trước** khi gán, ta không bị ảnh hưởng bởi thứ tự (một lỗi phổ biến khi viết tách rời, xem phần lỗi).

In [12]:
def delete_parallel(node):
    nxt = node.next
    node.val, node.next = nxt.val, nxt.next        # vế phải tính xong rồi mới gán


for values, idx, expected in examples:
    head = from_list(values)
    delete_parallel(node_at(head, idx))
    print(f"head={values}, xóa vị trí {idx} -> {to_list(head)} (kỳ vọng {expected})")

head=[4, 5, 1, 9], xóa vị trí 1 -> [4, 1, 9] (kỳ vọng [4, 1, 9])
head=[4, 5, 1, 9], xóa vị trí 2 -> [4, 5, 9] (kỳ vọng [4, 5, 9])


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 5 — Có `head`: dùng nút giả `dummy` để xóa mà không cần trường hợp đặc biệt

**Giải thích.** Để thấy cách xóa chính tắc đầy đủ khi có `head` (kể cả xóa nút đầu hoặc nút cuối): đặt `dummy.next = head`; tìm `prev` bắt đầu từ `dummy`; nối `prev.next = prev.next.next`; trả `dummy.next`. Cách này **không** áp dụng được cho đề (không có `head`) nhưng giúp hiểu vì sao đề phải dùng mẹo.

In [13]:
def delete_with_dummy(head, node):
    dummy = ListNode(0)
    dummy.next = head
    prev = dummy
    while prev.next is not node:
        prev = prev.next
    prev.next = prev.next.next                   # kể cả khi node là head hoặc đuôi
    return dummy.next


for values, idx, expected in examples:
    head = from_list(values)
    head = delete_with_dummy(head, node_at(head, idx))
    print(f"head={values}, xóa vị trí {idx} -> {to_list(head)} (kỳ vọng {expected})")
head = from_list([4, 5, 1, 9])
print("xóa đuôi:", to_list(delete_with_dummy(head, node_at(head, 3))))
head = from_list([4, 5, 1, 9])
print("xóa đầu :", to_list(delete_with_dummy(head, head)))

head=[4, 5, 1, 9], xóa vị trí 1 -> [4, 1, 9] (kỳ vọng [4, 1, 9])
head=[4, 5, 1, 9], xóa vị trí 2 -> [4, 5, 9] (kỳ vọng [4, 5, 9])
xóa đuôi: [4, 5, 1]
xóa đầu : [5, 1, 9]


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

> Cách 2 là cách nộp bài. Cách 4 viết gọn hơn. Cách 3 minh họa tư tưởng. Cách 1 và 5 cần `head`, dùng làm chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 2 trên `4 → 5 → 1 → 9` với `node` là nút `5`, in danh sách (kèm "địa chỉ" ngắn của từng nút để thấy nút nào bị bỏ qua), và Cách 3 trên `node` là nút `5`.

In [14]:
def show(head, label):
    parts, cur = [], head
    while cur:
        parts.append(f"{cur.val}@{id(cur) % 10000:04d}")
        cur = cur.next
    print(f"{label:<34}", " -> ".join(parts))


head = from_list([4, 5, 1, 9])
node = node_at(head, 1)
show(head, "ban đầu (node = nút 5):")
node.val = node.next.val
show(head, "sau node.val = node.next.val:")
node.next = node.next.next
show(head, "sau node.next = node.next.next:")
assert to_list(head) == [4, 1, 9]

ban đầu (node = nút 5):            4@8832 -> 5@7104 -> 1@8928 -> 9@9024
sau node.val = node.next.val:      4@8832 -> 1@7104 -> 1@8928 -> 9@9024
sau node.next = node.next.next:    4@8832 -> 1@7104 -> 9@9024


In [15]:
def trace_shift(values, idx):
    head = from_list(values)
    node = node_at(head, idx)
    show(head, f"ban đầu (xóa vị trí {idx}):")
    cur = node
    step = 0
    while cur.next.next is not None:
        cur.val = cur.next.val
        cur = cur.next
        step += 1
        show(head, f"  dịch trái, bước {step}:")
    cur.val = cur.next.val
    cur.next = None
    show(head, "  cắt đuôi:")
    return to_list(head)


assert trace_shift([4, 5, 1, 9], 1) == [4, 1, 9]

ban đầu (xóa vị trí 1):            4@7392 -> 5@7584 -> 1@9120 -> 9@9216
  dịch trái, bước 1:               4@7392 -> 1@7584 -> 1@9120 -> 9@9216
  cắt đuôi:                        4@7392 -> 1@7584 -> 9@9120


Bảng tóm tắt (Cách 2, xóa nút `5`):

| bước | danh sách (giá trị) | ghi chú |
|---|---|---|
| ban đầu | `4 → 5 → 1 → 9` | `node` là nút thứ 2 |
| `node.val = node.next.val` | `4 → 1 → 1 → 9` | nút thứ 2 mượn giá trị `1` |
| `node.next = node.next.next` | `4 → 1 → 9` | nút thứ 3 (bản sao thừa) bị bỏ qua |

Các "địa chỉ" ở trace cho thấy nút ở vị trí thứ 2 vẫn là **cùng một đối tượng** ban đầu (nhưng đã đổi giá trị); nút thứ 3 ban đầu là nút bị loại khỏi danh sách.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2. Đúng class `Solution`, đúng method `deleteNode(self, node)` (không trả về gì: sửa `node` tại chỗ).

In [16]:
class Solution:
    def deleteNode(self, node):
        """
        :type node: ListNode
        :rtype: void Do not return anything, modify node in-place instead.
        """
        node.val = node.next.val          # mượn giá trị của nút kế tiếp
        node.next = node.next.next        # rồi bỏ qua nút kế tiếp


for values, idx, expected in examples:
    head = from_list(values)
    result = Solution().deleteNode(node_at(head, idx))
    assert result is None and to_list(head) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Trạng thái nhìn từ ngoài:** người chấm chỉ nhìn **dãy giá trị** từ `head` tới cuối. Gọi dãy ban đầu là `v₀, ..., vᵢ, vᵢ₊₁, ..., vₙ₋₁` với `node` ở vị trí `i` (`i < n − 1`).
- **Sau `node.val = node.next.val`:** nút ở vị trí `i` chứa `vᵢ₊₁`; nút ở vị trí `i + 1` vẫn chứa `vᵢ₊₁` (bản sao). Dãy: `..., vᵢ₋₁, vᵢ₊₁, vᵢ₊₁, vᵢ₊₂, ...`.
- **Sau `node.next = node.next.next`:** nút vị trí `i` nối thẳng tới nút ban đầu ở `i + 2`: dãy `..., vᵢ₋₁, vᵢ₊₁, vᵢ₊₂, ...` — chính là dãy ban đầu bỏ đi `vᵢ`.
- **Điều kiện `node` không phải đuôi:** đảm bảo `node.next` không rỗng, để có giá trị để mượn. (Nếu `node` là đuôi, không thể xóa nó bằng cách này vì không thể làm cho nút trước nó ngừng trỏ tới nó.)
- **Giá trị phân biệt:** đề đảm bảo để người chấm xác định nút nào bị xóa bằng giá trị; thuật toán không dùng điều này.
- **Cách 3 đúng** vì mỗi lần dịch là "mượn giá trị nút kế" liên tiếp; kết quả cuối cùng cũng là dãy giá trị bỏ `vᵢ`. **Cách 4** giống Cách 2 về hiệu quả.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút trong danh sách và `m` là số nút từ `node` đến cuối.

- **Cách 1 (xóa thông thường, có `head`):** Time `O(n)`, Space `O(1)`.
- **Cách 2 (sao chép nút kế tiếp):** Time **`O(1)`**, Space **`O(1)`**.
- **Cách 3 (dịch về trái đến cuối):** Time `O(m)`, Space `O(1)`.
- **Cách 4 (gán song song):** Time `O(1)`, Space `O(1)`.
- **Cách 5 (có `head` và `dummy`):** Time `O(n)`, Space `O(1)`.
- Trường hợp trung bình và xấu nhất như nhau cho mọi cách.

## 12. Bộ test

Vì hàm sửa danh sách tại chỗ, mỗi lần kiểm tra ta **dựng lại danh sách mới**, chọn nút theo vị trí, gọi hàm rồi so sánh với `values` bỏ phần tử tại vị trí đó (oracle: cắt list Python, **không** dùng bất kỳ thao tác con trỏ nào). Cách 1 và Cách 5 nhận thêm `head` nên ta bọc lại cho thống nhất giao diện.

In [17]:
sol = Solution()


def wrap_no_head(fn):
    def run(head, node):
        fn(node)
        return head
    return run


def wrap_head(fn):
    def run(head, node):
        return fn(head, node)
    return run


impls = {
    "with_head": wrap_head(delete_with_head),
    "copy_next": wrap_no_head(delete_copy_next),
    "shift_left": wrap_no_head(delete_shift_left),
    "parallel": wrap_no_head(delete_parallel),
    "dummy": wrap_head(delete_with_dummy),
    "submit": wrap_no_head(sol.deleteNode),
}
NO_HEAD = {"copy_next", "shift_left", "parallel", "submit"}


def oracle(values, idx):
    return values[:idx] + values[idx + 1:]


def run_case(values, idx):
    expected = oracle(values, idx)
    for name, fn in impls.items():
        head = from_list(values)
        node = node_at(head, idx)
        head = fn(head, node)
        assert to_list(head) == expected, (name, values, idx, to_list(head), expected)
    return expected


for values, idx, expected in examples:
    assert run_case(values, idx) == expected
print("PASS ví dụ của đề (mọi cách; oracle là cắt list Python)")

PASS ví dụ của đề (mọi cách; oracle là cắt list Python)


### 12.1. Các trường hợp biên

In [18]:
# Đề: 2 <= n, node không phải đuôi -> idx từ 0 đến n - 2
assert run_case([1, 2], 0) == [2]                          # n = 2 nhỏ nhất, xóa nút đầu
assert run_case([1, 2, 3], 0) == [2, 3]                    # xóa nút đầu (là head)
assert run_case([1, 2, 3], 1) == [1, 3]                    # xóa nút giữa
assert run_case([1, 2, 3], 1) == [1, 3]
assert run_case([-1000, 1000], 0) == [1000]                # giá trị biên
assert run_case([0, -1, 5, 6], 2) == [0, -1, 6]            # nút áp cuối (idx = n - 2)
assert run_case(list(range(1000)), 0) == list(range(1, 1000))        # n = 1000, xóa đầu
assert run_case(list(range(1000)), 998) == list(range(998)) + [999]  # n = 1000, nút áp cuối
assert run_case(list(range(1000)), 500) == list(range(500)) + list(range(501, 1000))
print("PASS các trường hợp biên: n = 2, xóa nút đầu, nút giữa, nút áp cuối, n = 1000")

PASS các trường hợp biên: n = 2, xóa nút đầu, nút giữa, nút áp cuối, n = 1000


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(237)

# Tính chất 1 (mẹo Cách 2): các nút ĐỨNG SAU nút kế tiếp giữ nguyên danh tính (cùng đối tượng);
# các nút đứng TRƯỚC node giữ nguyên danh tính và giá trị; nút kế tiếp cũ bị loại khỏi danh sách.
for _ in range(300):
    n = random.randint(2, 30)
    values = random.sample(range(-1000, 1001), n)
    idx = random.randint(0, n - 2)
    head = from_list(values)
    nodes_before = []
    cur = head
    while cur:
        nodes_before.append(cur)
        cur = cur.next
    Solution().deleteNode(nodes_before[idx])
    nodes_after = []
    cur = head
    while cur:
        nodes_after.append(cur)
        cur = cur.next
    assert len(nodes_after) == n - 1
    assert nodes_after[:idx + 1] == nodes_before[:idx + 1]            # đến và gồm cả node: cùng đối tượng
    assert nodes_after[idx + 1:] == nodes_before[idx + 2:]            # phần sau: bỏ nút kế tiếp cũ
    assert nodes_before[idx + 1] not in nodes_after                   # nút kế tiếp cũ bị loại
print("PASS tính chất: độ dài giảm 1, danh tính các nút còn lại được bảo toàn")

# Tính chất 2: xóa lần lượt nhiều nút (luôn không phải đuôi) cho đúng kết quả oracle
for _ in range(200):
    n = random.randint(3, 20)
    values = random.sample(range(-1000, 1001), n)
    head = from_list(values)
    expected = values[:]
    for _step in range(random.randint(1, n - 1)):
        idx = random.randint(0, len(expected) - 2)
        Solution().deleteNode(node_at(head, idx))
        expected = oracle(expected, idx)
        assert to_list(head) == expected
print("PASS tính chất: xóa liên tiếp nhiều nút")

# Phản ví dụ: gán lại biến cục bộ không đổi danh sách
def sai_gan_lai_bien(node):
    node = node.next                         # chỉ đổi biến cục bộ

head = from_list([4, 5, 1, 9])
sai_gan_lai_bien(node_at(head, 1))
print("sai (gán lại biến):", to_list(head), "| đúng phải là [4, 1, 9]")
assert to_list(head) == [4, 5, 1, 9]

PASS tính chất: độ dài giảm 1, danh tính các nút còn lại được bảo toàn
PASS tính chất: xóa liên tiếp nhiều nút
sai (gán lại biến): [4, 5, 1, 9] | đúng phải là [4, 1, 9]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
for trial in range(2000):
    n = random.randint(2, 40)
    values = random.sample(range(-1000, 1001), n)
    idx = random.randint(0, n - 2)
    run_case(values, idx)                     # run_case so sánh từng cách với oracle (cắt list)
for trial in range(20):                       # danh sách dài (n tới 1000)
    n = random.randint(500, 1000)
    values = random.sample(range(-1000, 1001), n)
    run_case(values, random.randint(0, n - 2))
print("PASS 2020 danh sách ngẫu nhiên: mọi cách khớp oracle")

PASS 2020 danh sách ngẫu nhiên: mọi cách khớp oracle


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Gán lại biến `node` thay vì sửa đối tượng

In [21]:
def sai_gan_lai(node):
    node = node.next                          # chỉ đổi biến cục bộ


head = from_list([4, 5, 1, 9])
sai_gan_lai(node_at(head, 1))
print("sai:", to_list(head), "| đúng:", [4, 1, 9])

sai: [4, 5, 1, 9] | đúng: [4, 1, 9]


**Vì sao sai:** `node = node.next` làm biến cục bộ `node` trỏ sang nút khác, **không** thay đổi đối tượng nút hay con trỏ của nút đứng trước. Danh sách vẫn y nguyên. Muốn thay đổi danh sách phải **ghi vào trường** của đối tượng (`node.val = ...`, `node.next = ...`).

### Lỗi 2: Chỉ nối `node.next = node.next.next` (xóa nhầm nút kế tiếp)

In [22]:
def sai_chi_noi(node):
    node.next = node.next.next                # bỏ qua nút KẾ TIẾP, nhưng node vẫn giữ giá trị cũ


head = from_list([4, 5, 1, 9])
sai_chi_noi(node_at(head, 1))
print("sai:", to_list(head), "| đúng:", [4, 1, 9])

sai: [4, 5, 9] | đúng: [4, 1, 9]


**Vì sao sai:** lệnh đó xóa nút **sau** `node` (giá trị `1`), còn `node` (giá trị `5`) vẫn ở lại: kết quả `[4, 5, 9]` thay vì `[4, 1, 9]`. Phải sao chép giá trị của nút kế tiếp vào `node` **trước** (hoặc cùng lúc).

### Lỗi 3: Sai thứ tự hai lệnh gán

In [23]:
def sai_thu_tu(node):
    node.next = node.next.next                # nối trước...
    node.val = node.next.val                  # ...rồi mới chép: lúc này node.next đã là nút thứ 2 sau node


head = from_list([4, 5, 1, 9])
sai_thu_tu(node_at(head, 1))
print("sai:", to_list(head), "| đúng:", [4, 1, 9])

sai: [4, 9, 9] | đúng: [4, 1, 9]


**Vì sao sai:** sau `node.next = node.next.next`, `node.next` đã nhảy sang nút `9`; câu sau chép giá trị `9` vào `node`, cho `[4, 9, 9]`. Phải chép `val` trước khi nối, hoặc lưu `nxt = node.next` rồi dùng gán song song (Cách 4).

### Lỗi 4: Nghĩ rằng có thể xóa cả nút đuôi bằng cách này

In [24]:
head = from_list([4, 5, 1, 9])
tail = node_at(head, 3)
try:
    Solution().deleteNode(tail)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'val'


**Vì sao sai:** ở nút đuôi `node.next` là `None`, nên không có giá trị để mượn (`None.val` gây `AttributeError`). Đây là lý do đề đảm bảo `node` **không phải đuôi**: với danh sách đơn và chỉ có `node`, không có cách nào làm cho nút đứng trước đuôi ngừng trỏ tới đuôi.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "xóa một nút khi **chỉ có tham chiếu tới nút đó**" (không có `head`/`prev`): dùng mẹo **mượn nội dung nút kế tiếp rồi bỏ qua nút kế tiếp**.
- Mẹo này **thay đổi nội dung** các nút chứ không bỏ chính đối tượng `node`; nếu nơi khác giữ tham chiếu tới nút kế tiếp cũ, chúng sẽ thấy nút đó bị loại. Dùng khi đề chấp nhận (như bài này).
- Nhớ: gán lại biến không sửa danh sách; phải ghi vào **trường** `val`/`next`.
- Có `head`: dùng `dummy` để xóa nút đầu/đuôi/giữa đồng nhất.
- Cẩn thận **thứ tự** các lệnh gán liên quan đến `node.next`; dùng gán song song khi cần.
- Khi test hàm sửa tại chỗ, **dựng danh sách mới cho mỗi lần chạy** và so sánh với oracle trên list Python.

## 15. Biến thể của bài

1. **Xóa mọi nút có giá trị `val`** khi có `head` (LeetCode 203).
2. **Xóa nút thứ `n` tính từ cuối** (LeetCode 19).
3. **Danh sách liên kết đôi:** xóa một nút chỉ cần có nút đó, `O(1)`.
4. **Xóa nút trùng lặp** trong danh sách đã sắp (LeetCode 83).
5. **Hoán đổi giá trị thay vì nối lại** (khi bị cấm sửa `next`).

In [25]:
def remove_elements(head, val):
    """Biến thể 1: xóa mọi nút có giá trị val (có head, dùng dummy)."""
    dummy = ListNode(0)
    dummy.next = head
    prev = dummy
    while prev.next:
        if prev.next.val == val:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next


def remove_nth_from_end(head, n):
    """Biến thể 2: hai con trỏ cách nhau n bước."""
    dummy = ListNode(0)
    dummy.next = head
    fast = slow = dummy
    for _ in range(n):
        fast = fast.next
    while fast.next:
        fast, slow = fast.next, slow.next
    slow.next = slow.next.next
    return dummy.next


class DNode:
    def __init__(self, x):
        self.val = x
        self.prev = None
        self.next = None


def delete_doubly(node):
    """Biến thể 3: danh sách liên kết đôi, xóa đúng nút đó (kể cả đuôi) trong O(1)."""
    if node.prev:
        node.prev.next = node.next
    if node.next:
        node.next.prev = node.prev


def remove_duplicates_sorted(head):
    """Biến thể 4."""
    cur = head
    while cur and cur.next:
        if cur.val == cur.next.val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return head


assert to_list(remove_elements(from_list([1, 2, 6, 3, 6]), 6)) == [1, 2, 3]
assert to_list(remove_elements(from_list([7, 7, 7]), 7)) == []
assert to_list(remove_nth_from_end(from_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_list(remove_nth_from_end(from_list([1]), 1)) == []
nodes = [DNode(v) for v in [1, 2, 3, 4]]
for a, b in zip(nodes, nodes[1:]):
    a.next, b.prev = b, a
delete_doubly(nodes[3])          # xóa đuôi được vì có prev
delete_doubly(nodes[1])
vals, cur = [], nodes[0]
while cur:
    vals.append(cur.val)
    cur = cur.next
assert vals == [1, 3]
assert to_list(remove_duplicates_sorted(from_list([1, 1, 2, 3, 3]))) == [1, 2, 3]
random.seed(15)
for _ in range(200):
    values = [random.randint(0, 4) for _ in range(random.randint(0, 12))]
    val = random.randint(0, 4)
    assert to_list(remove_elements(from_list(values), val)) == [v for v in values if v != val]
print("PASS biến thể: xóa theo giá trị, từ cuối, danh sách đôi, trùng lặp")

PASS biến thể: xóa theo giá trị, từ cuối, danh sách đôi, trùng lặp


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Xóa thông thường (có `head`) | tìm `prev`, nối `prev.next = node.next` | `O(n)` | `O(1)` | chuẩn đối chiếu; cần `head` |
| 2. Mượn nút kế tiếp | `node.val = node.next.val; node.next = node.next.next` | `O(1)` | `O(1)` | **cách nộp bài** |
| 3. Dịch giá trị về trái | ghi đè từng nút bằng nút kế, cắt đuôi | `O(m)` | `O(1)` | minh họa tư tưởng |
| 4. Gán song song | `node.val, node.next = nxt.val, nxt.next` | `O(1)` | `O(1)` | viết gọn, tránh sai thứ tự |
| 5. `dummy` (có `head`) | `dummy.next = head`, xóa đồng nhất | `O(n)` | `O(1)` | khi có `head` và cần xử lý nút đầu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `to_list(from_list([1, 2, 3]))`, `node_at(from_list([1, 2, 3]), 2).val`, danh sách sau khi gọi `Solution().deleteNode` trên nút `2` của `[1, 2, 3]`, và trên nút `1` của `[1, 2, 3]`.

**Bài 2 (tính tay).** Danh sách `7 → 8 → 9 → 10`: vẽ lại danh sách (giá trị) sau mỗi lệnh của Cách 2 khi `node` là nút `8`, rồi khi `node` là nút `7`. Danh sách cuối là gì?

**Bài 3 (điền TODO).** Hoàn thành `delete_todo(node)` theo Cách 2.

In [26]:
def delete_todo(node):
    # TODO 1: chép giá trị của nút kế tiếp vào node
    # TODO 2: cho node nối thẳng tới nút sau nút kế tiếp
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `delete_shift2(node)` theo Cách 3 (dịch giá trị về trái rồi cắt đuôi) mà không nhìn code mẫu. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `delete_k_after(node, k)` xóa nút **cách `node` `k` vị trí về phía sau** (`k ≥ 1`) khi chỉ có `node` (`k ≥ 1`, nút đích tồn tại; nút đích có thể là đuôi vì ta nắm được nút đứng ngay trước nó). (b) Viết `delete_all_after(node)` cắt toàn bộ phần đứng sau `node` (nút `node` trở thành đuôi).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `to_list` đổi danh sách thành list Python; mẹo mượn giá trị của nút kế tiếp. Bài 2: viết ba trạng thái: ban đầu, sau chép giá trị, sau bỏ qua.
- **Mức 2 (vừa):** Bài 1: `[1, 2, 3]`, `3`, `[1, 3]`, `[2, 3]`. Bài 2: `7 → 8 → 9 → 10`, node = 8: sau chép `7 → 9 → 9 → 10`, sau nối `7 → 9 → 10`. Tiếp tục với node = nút đầu `7`: chép `9 → 9 → 10`, nối `9 → 10`. Bài 3: hai TODO trong comment.
- **Mức 3 (sâu):** Bài 4: vòng lặp `while cur.next.next` ghi `cur.val = cur.next.val`, rồi xử lý nút áp cuối và đặt `cur.next = None`. Bài 5(a): đi tới nút trước nút đích `k − 1` bước từ `node`, rồi `prev.next = prev.next.next` (cần nút trước, có thể lấy được từ `node`). Bài 5(b): `node.next = None`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert to_list(from_list([1, 2, 3])) == [1, 2, 3]
assert node_at(from_list([1, 2, 3]), 2).val == 3
h = from_list([1, 2, 3]); Solution().deleteNode(node_at(h, 1)); assert to_list(h) == [1, 3]
h = from_list([1, 2, 3]); Solution().deleteNode(node_at(h, 0)); assert to_list(h) == [2, 3]

# Bài 2
h = from_list([7, 8, 9, 10]); n = node_at(h, 1)
n.val = n.next.val; assert to_list(h) == [7, 9, 9, 10]
n.next = n.next.next; assert to_list(h) == [7, 9, 10]
n0 = node_at(h, 0)
n0.val = n0.next.val; assert to_list(h) == [9, 9, 10]
n0.next = n0.next.next; assert to_list(h) == [9, 10]

# Bài 3
def delete_todo_ans(node):
    node.val = node.next.val
    node.next = node.next.next

for values, idx, expected in examples:
    h = from_list(values)
    delete_todo_ans(node_at(h, idx))
    assert to_list(h) == expected

# Bài 4
def delete_shift2(node):
    cur = node
    while cur.next.next is not None:
        cur.val = cur.next.val
        cur = cur.next
    cur.val = cur.next.val
    cur.next = None

for values, idx, expected in examples:
    h = from_list(values)
    delete_shift2(node_at(h, idx))
    assert to_list(h) == expected

# Bài 5(a), 5(b)
def delete_k_after(node, k):
    prev = node
    for _ in range(k - 1):
        prev = prev.next
    prev.next = prev.next.next          # bỏ nút cách node đúng k vị trí

def delete_all_after(node):
    node.next = None

h = from_list([1, 2, 3, 4, 5]); delete_k_after(node_at(h, 1), 2); assert to_list(h) == [1, 2, 3, 5]
h = from_list([1, 2, 3, 4, 5]); delete_k_after(node_at(h, 0), 1); assert to_list(h) == [1, 3, 4, 5]
h = from_list([1, 2, 3, 4, 5]); delete_all_after(node_at(h, 2)); assert to_list(h) == [1, 2, 3]
random.seed(19)
for _ in range(200):
    n = random.randint(3, 15)
    values = random.sample(range(100), n)
    i = random.randint(0, n - 3)
    k = random.randint(1, n - 1 - i)
    h = from_list(values)
    delete_k_after(node_at(h, i), k)
    assert to_list(h) == values[:i + k] + values[i + k + 1:]
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xóa nút khi chỉ có chính nút đó)

```python
class Solution:
    def deleteNode(self, node):
        node.val = node.next.val
        node.next = node.next.next
```

**Checklist**

- [ ] Hiểu rằng ta **không có `head`/`prev`**, nên phải **mượn nội dung** của nút kế tiếp?
- [ ] Chép `node.val = node.next.val` **trước**, rồi mới `node.next = node.next.next`?
- [ ] Ghi vào **trường** của đối tượng (`node.val`, `node.next`), không gán lại biến `node`?
- [ ] Nhớ điều kiện: `node` **không phải đuôi** (nên `node.next` không rỗng)?
- [ ] Hàm **không trả về gì** (sửa tại chỗ)?
- [ ] Đã thử: n = 2, xóa nút đầu, nút giữa, nút áp cuối?
- [ ] Đã đối chiếu với oracle (cắt list Python) trên danh sách ngẫu nhiên, dựng danh sách mới cho mỗi lần?
- [ ] Skeleton đúng `class Solution` / `def deleteNode(self, node)`?